In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/FC26_20250921.csv")
df.drop(columns=["fifa_version", "fifa_update", "fifa_update_date", "player_face_url", "work_rate", "player_url"], inplace=True)

df_draft = df[df['overall'] >= 75].copy()

faixas = [
        (df_draft['overall'] >= 87),
        (df_draft['overall'] >= 84) & (df_draft['overall'] < 87),
        (df_draft['overall'] >= 80) & (df_draft['overall'] < 84),
        (df_draft['overall'] < 80)
]
    
pesos = [5, 15, 40, 80]
df_draft['weight'] = np.select(faixas, pesos, default=0)

/tmp/ipykernel_10792/387547489.py:4: DtypeWarning: Columns (0: player_tags) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/FC26_20250921.csv")


In [7]:
from draft import generate_draft_pack
from bot import Bot
from eafc_utils import f

print("===========================================\n INICIANDO SIMULADOR DE DRAFT EA FC \n===========================================")

formacao_433 = ['ST', 'LW', 'RW', 'CM', 'CM', 'CM', 'LB', 'CB', 'CB', 'RB', 'GK']
meu_time = []
ids_escolhidos = set()

NUMERO_DE_ROLLOUTS = 1000

meu_bot = Bot(mode="greedy_f", num_rollouts=NUMERO_DE_ROLLOUTS)

for rodada, posicao_alvo in enumerate(formacao_433, start=1):
    texto = f"\n--- RODADA {rodada}/11 | Buscando: {posicao_alvo} ---"
    if rodada == 1:
        texto = "Rodada inicial. Selecionando capitães."
    print(texto)

    pacote = generate_draft_pack(df_draft, rodada, posicao_alvo, ids_escolhidos)
    opcoes = pacote.reset_index(drop=True)
    opcoes['choosen_position'] = posicao_alvo
    print(opcoes[['short_name', 'overall', 'club_name', 'league_name', 'nationality_name']])

    df_meu_time = pd.DataFrame(meu_time) if meu_time else pd.DataFrame(columns=df_draft.columns)

    posicoes_restantes = formacao_433[rodada:]

    indice_escolhido = meu_bot.choose(
        options=opcoes,
        current_squad=df_meu_time,
        db=df_draft,
        current_round=rodada,
        remaining_positions=posicoes_restantes
    )

    carta_escolhida = opcoes.iloc[indice_escolhido].copy()
    # Se o jogador nao joga na posicao do slot atual, redireciona para sua posicao natural
    # Troca também o slot natural pelo slot atual, evitando posições duplicatas
    posicao_natural = str(carta_escolhida['player_positions']).split(',')[0].strip()
    if posicao_natural != posicao_alvo and posicao_natural in formacao_433[rodada:]:
        idx = formacao_433.index(posicao_natural, rodada)
        formacao_433[idx] = posicao_alvo
        carta_escolhida['choosen_position'] = posicao_natural
    
    meu_time.append(carta_escolhida)
    ids_escolhidos.add(carta_escolhida['player_id'])

    print(f">>> ESCOLHA DO BOT: {carta_escolhida['short_name']} (OVR {carta_escolhida['overall']})")

print("\n===========================================")
print(" DRAFT CONCLUÍDO! SEU ELENCO FINAL: ")
print("===========================================")
df_final = pd.DataFrame(meu_time)
print(df_final[['short_name', 'overall', 'league_name', 'nationality_name', 'club_name']])
nota_final = int(f(df_final))
print(f"\nNOTA FINAL DO TIME:", nota_final)

 INICIANDO SIMULADOR DE DRAFT EA FC 
Rodada inicial. Selecionando capitães.
  short_name  overall        club_name     league_name nationality_name
0      Rodri       90  Manchester City  Premier League            Spain
1   J. Oblak       88  Atlético Madrid         La Liga         Slovenia
2   M. Salah       91        Liverpool  Premier League            Egypt
3   F. Wirtz       89        Liverpool  Premier League          Germany
4    B. Saka       88          Arsenal  Premier League          England
>>> ESCOLHA DO BOT: M. Salah (OVR 91)

--- RODADA 2/11 | Buscando: LW ---
     short_name  overall                club_name                 league_name  \
0     C. Arango       78     San Jose Earthquakes         Major League Soccer   
1   Richarlison       78        Tottenham Hotspur              Premier League   
2      S. Villa       75  Independiente Rivadavia  Liga Profesional de Fútbol   
3  G. Raspadori       78          Atlético Madrid                     La Liga   
4        M. C